In [ ]:
src\config\settings.py  #<----------------------

In [ ]:
…
 INSTALLED_APPS = [
    "ventas", #<------------ Para la nueva app de ventas
    "pages.apps.PagesConfig",
    "products.apps.ProductsConfig", 
    "ecommerce.apps.EcommerceConfig",
    "base.apps.BaseConfig",  
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
]
…


In [ ]:
src\ventas\views.py:  #<----------------------

from django.shortcuts import render, redirect
from django.http import HttpResponse
from .models import Producto

from django.views.decorators.http import require_POST

# Create your views here.

# Para crear la lista de productos
def lista_productos(request):
    productos = Producto.objects.all()
    return render(request, 'ventas/productos.html', {
        'productos': productos
    })

# Para agregar productos y al mismo tiempo redirigir al carrito
def agregar_al_carrito(request, producto_id):
    carrito = request.session.get('carrito', {})

    producto_id = str(producto_id)

    if producto_id in carrito:
        carrito[producto_id] += 1
    else:
        carrito[producto_id] = 1

    request.session['carrito'] = carrito

    return redirect('ventas:carrito') 

# Para que puedas ver tu crrito de productos Ver carrito
def ver_carrito(request):
    carrito = request.session.get('carrito', {})
    productos = Producto.objects.filter(id__in=carrito.keys())

    items = []
    total = 0

    for producto in productos:
        cantidad = carrito[str(producto.id)]
        subtotal = producto.precio * cantidad
        total += subtotal

        items.append({
            'producto': producto,
            'cantidad': cantidad,
            'subtotal': subtotal
        })

    return render(request, 'ventas/carrito.html', {
        'items': items,
        'total': total
    })

# Para simular el procesar del pedido y redirigir
def procesar_pedido(request):
    request.session['carrito'] = {}
    return redirect('ventas:confirmacion')

# 5. Página final
def confirmacion(request):
    return render(request, 'ventas/confirmacion.html')

# para poder crear productos desde la interfas
@require_POST
def crear_producto(request):
    nombre = request.POST.get('nombre')
    precio = request.POST.get('precio')

    if nombre and precio:
        Producto.objects.create(
            nombre=nombre,
            precio=precio
        )

    return redirect('ventas:productos')


@require_POST
def eliminar_producto(request, producto_id):
    producto = Producto.objects.filter(id=producto_id).first()

    if producto:
        producto.delete()

    return redirect('ventas:productos')

In [ ]:
src\ventas\models.py:  #<----------------------

In [ ]:

from django.db import models

from django.db import models

# Create your models here.

class Producto(models.Model):
    nombre = models.CharField(max_length=100)
    precio = models.DecimalField(max_digits=10, decimal_places=2)

    def __str__(self):
        return self.nombre


In [ ]:
src\ventas\urls.py: #<----------------------

In [ ]:

from django.urls import path
from . import views

app_name = 'ventas'

urlpatterns = [
    path('', views.lista_productos, name='productos'), # Pagina principal de los productos  
    path('agregar/<int:producto_id>/', views.agregar_al_carrito, name='agregar'), # Para agregar o productos al carrito
    path('carrito/', views.ver_carrito, name='carrito'), # Para ver los productos agregado al carrito
    path('procesar/', views.procesar_pedido, name='procesar'), # Para procesar el producto
    path('confirmacion/', views.confirmacion, name='confirmacion'), # Para simular la confirmacion 
    path('crear/', views.crear_producto, name='crear'), # para crear el producto
    path('eliminar/<int:producto_id>/', views.eliminar_producto, name='eliminar'), # Para eliminar un producto expecifico
]

In [ ]:
src\ventas\templates\ventas\productos.html:  #<----------------------

In [ ]:

<h1>🛒 Productos EBAC 🛒</h1>

<!-- Este es el formulario para crear los productos-->
<h2>Agregar nuevo producto EBAC</h2>

<form method="POST" action="{% url 'ventas:crear' %}">
    {% csrf_token %}
    <input type="text" name="nombre" placeholder="Nombre del producto" required>
    <input type="number" name="precio" placeholder="Precio" required>
    <button type="submit">Crear producto de EBAC</button>
</form>

<hr>

<!--  LISTA DE PRODUCTOS -->
<ul>
{% for producto in productos %}
    <li>
        {{ producto.nombre }} - ${{ producto.precio }}
        <a href="{% url 'ventas:agregar' producto.id %}">
             Agregar al carrito de EBAC
        </a>
    </li>
{% endfor %}
</ul>

<hr>

<a href="{% url 'ventas:carrito' %}">🛒 Ver carrito de EBAC</a>

In [ ]:
src\ventas\templates\ventas\carrito.html:  #<----------------------

In [ ]:

<h1>Carrito EBAC</h1>

{% if items %}
    <ul>
    {% for item in items %}
        <li>
            {{ item.producto.nombre }} |
            Cantidad: {{ item.cantidad }} |
            Subtotal: ${{ item.subtotal }}
        </li>
    {% endfor %}
    </ul>

    <h3>Total productos EBAC: ${{ total }}</h3>

    <!-- Para simular confirmacion de compra -->
    <a href="{% url 'ventas:procesar' %}">
        🛒 Confirmar compra de EBAC
    </a>

{% else %}
    <p>Carrito vacío EBAC</p>
{% endif %}

<br>
<a href="{% url 'ventas:productos' %}">Volver a productos principal de EBAC</a>

In [ ]:
src\ventas\templates\ventas\confirmacion.html:  #<----------------------

In [ ]:

<!DOCTYPE html>
<html>
<head>
    <title>Tienda EBAC</title>
</head>
<body>

<h1>Productos EBAC</h1>

<ul>
    {% for producto in productos %}
        <li>
            {{ producto.nombre }} - ${{ producto.precio }}
            <a href="{% url 'ventas:agregar' producto.id %}">
                Agregar al carrito de EBAC
            </a>
        </li>
    {% endfor %}
</ul>

<hr>

<h2>Carrito EBAC</h2>
                   
{% if items %}
    <ul>
        {% for item in items %}
            <li>
                {{ item.producto.nombre }} |
                Cantidad: {{ item.cantidad }} |
                Subtotal: ${{ item.subtotal }}
            </li>
        {% endfor %}
    </ul>

    <h3>Total: ${{ total }}</h3>

    <a href="{% url 'ventas:procesar' %}">Finalizar compra en EBAC</a>

{% else %}
    <p>Carrito de EBAC vacío</p>
{% endif %}

</body>
</html>